In [ ]:
import yfinance as yf
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import joblib

tickers = ["AAPL","MSFT","GOOGL","AMZN","TSLA","NVDA","BRK-B","^GSPC","^IXIC","^FTSE","^DJI"]
data = yf.download(tickers, start="2010-01-01", end="2026-03-13")
all_data = {}

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
for ticker in tickers:
    stock = yf.Ticker(ticker)
    data = stock.history(period="10y")

    if data.empty:
        print(ticker, "has no data, skipping")
        continue

    data = data.sort_index(ascending=True)  # oldest to newest
    data = data[~data.index.duplicated(keep="first")]
    data = data[(data["Close"] > 0) & (data["Volume"] > 0)]
    data["Ticker"] = ticker
    if len(data) < 250:
        print(ticker, "doesn't have enough history, skipping")
        continue

    data["MA20"] = data["Close"].rolling(20).mean()
    data["MA50"] = data["Close"].rolling(50).mean()
    data["MA200"] = data["Close"].rolling(200).mean()

    data["Return5"] = (data["Close"] / data["Close"].shift(5)) - 1
    data["Return20"] = (data["Close"] / data["Close"].shift(20)) - 1
    data["MA20/MA50"] = data["MA20"] / data["MA50"]
    data["Close/MA200"] = data["Close"] / data["MA200"]
    data["Volume/Volume_MA20"] = data["Volume"] / data["Volume"].rolling(20).mean()

    data["Target"] = (data["Close"].shift(-1) > data["Close"]).astype(int)
    data = data.dropna(subset=["Open","High","Low","Close","Volume","MA20", "MA50", "MA200"])
    data = data.reset_index()

    all_data[ticker] = data
combined_data = pd.concat(all_data.values(), ignore_index=True, axis=0, sort=False)
model = RandomForestClassifier(n_estimators=100, random_state=42)

features = [
    "Return5",
    "Return20",
    "MA20/MA50",
    "Close/MA200",
    "Volume/Volume_MA20",
]

combined_data["Date"] = pd.to_datetime(combined_data["Date"], utc=True)
combined_data["Date"] = combined_data["Date"].dt.tz_convert(None)
cutoff = combined_data["Date"].max() - pd.DateOffset(years=2)
train_data = combined_data[combined_data["Date"] < cutoff]
test_data = combined_data[combined_data["Date"] >= cutoff]

X_Train = train_data[features]
y_Train = train_data["Target"]

X_Test = test_data[features]
y_Test = test_data["Target"]

model.fit(X_Train, y_Train)

joblib.dump(model, "random_forest_model.joblib")

['random_forest_model.joblib']

In [ ]:
model = joblib.load("random_forest_model.joblib")

pred_probs = model.predict_proba(X_Test)[:, 1]  # probability price goes up
pred_class = model.predict(X_Test)        

predictions_df = combined_data.loc[X_Test.index].copy()

predictions_df["Actual"] = y_Test.values
predictions_df["Pred_Prob"] = pred_probs
predictions_df["Pred_Class"] = pred_class

latest = predictions_df.sort_values("Date").groupby("Ticker").tail(1)
print(latest[["Ticker","Date","Pred_Prob","Pred_Class"]])

      Ticker                Date  Pred_Prob  Pred_Class
23167  ^FTSE 2026-03-12 00:00:00       0.57           1
9263    AMZN 2026-03-13 04:00:00       0.35           0
16211  BRK-B 2026-03-13 04:00:00       0.58           1
11579   TSLA 2026-03-13 04:00:00       0.55           1
20842  ^IXIC 2026-03-13 04:00:00       0.37           0
13895   NVDA 2026-03-13 04:00:00       0.47           0
18526  ^GSPC 2026-03-13 04:00:00       0.74           1
6947   GOOGL 2026-03-13 04:00:00       0.65           1
4631    MSFT 2026-03-13 04:00:00       0.62           1
2315    AAPL 2026-03-13 04:00:00       0.64           1
25483   ^DJI 2026-03-13 04:00:00       0.63           1
